In [2]:
import pandas as pd

movies = pd.read_csv("data/movie.csv")
ratings = pd.read_csv("data/rating.csv")
tags = pd.read_csv("data/tag.csv")
link = pd.read_csv("data/link.csv")

print("Movies:", movies.shape)
print("Ratings:", ratings.shape)
print("Tags:", tags.shape)
print("Links:", link.shape)
movies.head()

Movies: (27278, 3)
Ratings: (20000263, 4)
Tags: (465564, 4)
Links: (27278, 3)


,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


In [3]:
from scipy.sparse import csr_matrix

user_ids = ratings["userId"].astype("category").cat.codes
movie_ids = ratings["movieId"].astype("category").cat.codes

sparse_matrix = csr_matrix(
    (ratings["rating"], (user_ids, movie_ids))
)

sparse_matrix

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 20000263 stored elements and shape (138493, 26744)>

In [4]:
from sklearn.neighbors import NearestNeighbors

model = NearestNeighbors(metric='cosine', algorithm='brute')

model.fit(sparse_matrix.T)

,"n_neighbors n_neighbors: int, default=5Number of neighbors to use by default for :meth:`kneighbors` queries.",5
,"radius radius: float, default=1.0Range of parameter space to use by default for :meth:`radius_neighbors`queries.",1.0
,"algorithm algorithm: {'auto', 'ball_tree', 'kd_tree', 'brute'}, default='auto'Algorithm used to compute the nearest neighbors:- 'ball_tree' will use :class:`BallTree`- 'kd_tree' will use :class:`KDTree`- 'brute' will use a brute-force search.- 'auto' will attempt to decide the most appropriate algorithm based on the values passed to :meth:`fit` method.Note: fitting on sparse input will override the setting ofthis parameter, using brute force.",'brute'
,"leaf_size leaf_size: int, default=30Leaf size passed to BallTree or KDTree. This can affect thespeed of the construction and query, as well as the memoryrequired to store the tree. The optimal value depends on thenature of the problem.",30
,"metric metric: str or callable, default='minkowski'Metric to use for distance computation. Default is ""minkowski"", whichresults in the standard Euclidean distance when p = 2. See thedocumentation of `scipy.spatial.distance`_ andthe metrics listed in:class:`~sklearn.metrics.pairwise.distance_metrics` for valid metricvalues.If metric is ""precomputed"", X is assumed to be a distance matrix andmust be square during fit. X may be a :term:`sparse graph`, in whichcase only ""nonzero"" elements may be considered neighbors.If metric is a callable function, it takes two arrays representing 1Dvectors as inputs and must return one value indicating the distancebetween those vectors. This works for Scipy's metrics, but is lessefficient than passing the metric name as a string.",'cosine'
,"p p: float (positive), default=2Parameter for the Minkowski metric fromsklearn.metrics.pairwise.pairwise_distances. When p = 1, this isequivalent to using manhattan_distance (l1), and euclidean_distance(l2) for p = 2. For arbitrary p, minkowski_distance (l_p) is used.",2
,"metric_params metric_params: dict, default=NoneAdditional keyword arguments for the metric function.",None
,"n_jobs n_jobs: int, default=NoneThe number of parallel jobs to run for neighbors search.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.",None


In [6]:
movie_ids = ratings["movieId"].astype("category")
movie_to_index = dict(zip(movie_ids.cat.categories, range(len(movie_ids.cat.categories))))
index_to_movie = dict(zip(range(len(movie_ids.cat.categories)), movie_ids.cat.categories))

In [7]:
rating_counts = ratings.groupby("movieId").size()

popular_movies = rating_counts[rating_counts > 500].index

movies_popular = movies[movies["movieId"].isin(popular_movies)]

In [8]:
def recommend(movie_title, n=10):

    # find movie in dataset
    match = movies[movies["title"].str.contains(movie_title, case=False, regex=False)]

    if match.empty:
        print("Movie not found in dataset.")
        return []

    movie_id = match["movieId"].iloc[0]

    # check if movie has ratings (exists in matrix)
    if movie_id not in movie_to_index:
        print("Movie exists but has no ratings in the dataset.")
        return []

    movie_idx = movie_to_index[movie_id]

    # find nearest neighbors
    distances, indices = model.kneighbors(
        sparse_matrix.T[movie_idx],
        n_neighbors=n+1
    )

    recommendations = []

    for i in indices.flatten()[1:]:
        neighbor_movie_id = index_to_movie[i]

        if neighbor_movie_id in popular_movies:
            title = movies[movies["movieId"] == neighbor_movie_id]["title"].values[0]
            recommendations.append(title)

    return recommendations[:n]

In [9]:
movie_stats = ratings.groupby("movieId").agg({
    "rating": ["mean", "count"]
})

movie_stats.columns = ["avg_rating", "rating_count"]
movie_stats = movie_stats.reset_index()

movie_stats = movie_stats.merge(movies, on="movieId")

movie_stats.sort_values("rating_count", ascending=False).head(10)

,movieId,avg_rating,rating_count,title,genres
293,296,4.174231,67310,Pulp Fiction (1994),Comedy|Crime|Drama|Thriller
352,356,4.029000,66172,Forrest Gump (1994),Comedy|Drama|Romance|War
315,318,4.446990,63366,"Shawshank Redemption, The (1994)",Crime|Drama
587,593,4.177057,63299,"Silence of the Lambs, The (1991)",Crime|Horror|Thriller
476,480,3.664741,59715,Jurassic Park (1993),Action|Adventure|Sci-Fi|Thriller
257,260,4.190672,54502,Star Wars: Episode IV - A New Hope (1977),Action|Adventure|Sci-Fi
108,110,4.042534,53769,Braveheart (1995),Action|Drama|War
583,589,3.931954,52244,Terminator 2: Judgment Day (1991),Action|Sci-Fi
2486,2571,4.187186,51334,"Matrix, The (1999)",Action|Sci-Fi|Thriller
523,527,4.310175,50054,Schindler's List (1993),Drama|War


In [10]:
def show_recommendations(title):
    print("Recommendations for:", title)
    print("----------------------------")

    recs = recommend(title)

    for i, r in enumerate(recs, 1):
        print(f"{i}. {r}")

In [11]:
recommend("Toy Story (1995)")

['Star Wars: Episode IV - A New Hope (1977)',
 'Independence Day (a.k.a. ID4) (1996)',
 'Star Wars: Episode VI - Return of the Jedi (1983)',
 'Toy Story 2 (1999)',
 'Back to the Future (1985)',
 'Forrest Gump (1994)',
 'Aladdin (1992)',
 'Mission: Impossible (1996)',
 'Jurassic Park (1993)',
 'Willy Wonka & the Chocolate Factory (1971)']

In [12]:
recommend("The Dark Knight (2011)")

[]

In [32]:
movies[movies["title"].str.contains("Dark", case=False)]

,movieId,title,genres
202,204,Under Siege 2: Dark Territory (1995),Action
1029,1049,"Ghost and the Darkness, The (1996)",Action|Adventure
1189,1215,Army of Darkness (1993),Action|Adventure|Comedy|Fantasy|Horror
1684,1748,Dark City (1998),Adventure|Film-Noir|Sci-Fi|Thriller
2056,2140,"Dark Crystal, The (1982)",Adventure|Fantasy
...,...,...,...
26379,126785,A Spell to Ward Off the Darkness (2013),Documentary
26416,126971,Carts of Darkness (2008),Documentary
26776,128695,The Dark Valley (2014),Western
27078,130219,The Dark Knight (2011),Action|Crime|Drama|Thriller


In [33]:
movies[movies["title"].str.contains("Dark Knight", case=False)]

,movieId,title,genres
12525,58559,"Dark Knight, The (2008)",Action|Crime|Drama|IMAX
18312,91529,"Dark Knight Rises, The (2012)",Action|Adventure|Crime|IMAX
19876,98124,"Batman: The Dark Knight Returns, Part 1 (2012)",Action|Animation|Sci-Fi
20307,99813,"Batman: The Dark Knight Returns, Part 2 (2013)",Action|Animation
21255,103454,Batman Unmasked: The Psychology of the Dark Kn...,Documentary
27078,130219,The Dark Knight (2011),Action|Crime|Drama|Thriller


In [34]:
movies[movies["title"].str.contains("Knight", case=False)]

,movieId,title,genres
166,168,First Knight (1995),Action|Drama|Romance
324,328,Tales from the Crypt Presents: Demon Knight (1...,Horror|Thriller
3470,3561,Stacy's Knights (1982),Drama
3528,3619,"Hollywood Knights, The (1980)",Comedy
3714,3805,Knightriders (1981),Action|Adventure|Drama
4204,4299,"Knight's Tale, A (2001)",Action|Comedy|Romance
4803,4899,Black Knight (2001),Adventure|Comedy|Fantasy
6057,6156,Shanghai Knights (2003),Action|Adventure|Comedy
6410,6520,Knights of the Round Table (1953),Adventure|Drama
6894,7006,Knight Moves (1992),Mystery|Thriller


In [35]:
recommend("The Dark Knight (2011)")

[]

In [36]:
rating_counts[rating_counts.index == 130219]

movieId
130219    1
dtype: int64

In [37]:
recommend("Toy Story (1995)")

['Star Wars: Episode IV - A New Hope (1977)',
 'Independence Day (a.k.a. ID4) (1996)',
 'Star Wars: Episode VI - Return of the Jedi (1983)',
 'Toy Story 2 (1999)',
 'Back to the Future (1985)',
 'Forrest Gump (1994)',
 'Aladdin (1992)',
 'Mission: Impossible (1996)',
 'Jurassic Park (1993)',
 'Willy Wonka & the Chocolate Factory (1971)']